# Einstein Summation and Index Notation — Exercises

Practice problems covering index notation, einsum operations, gradient derivation,
contraction optimisation, and attention mechanism implementation.

## Format

| Symbol | Meaning |
|--------|----------|
| ⭐ | Essential — core index notation skills |
| ⭐⭐ | Applied — gradients, einsum, optimisation |
| ⭐⭐⭐ | Advanced — full attention pass, production patterns |

## Exercise Map

| Section | Exercises | Level |
|---------|-----------|-------|
| Free & Dummy Indices | 1 | ⭐ |
| Kronecker Delta & Levi-Civita | 2 | ⭐ |
| Gradient Derivation | 3a, 3b, 3c | ⭐⭐ |
| Einsum Implementation | 4a, 4b, 4c | ⭐⭐ |
| Contraction Order | 5 | ⭐⭐ |
| Softmax Gradient | 6 | ⭐⭐ |
| Symmetry Analysis | 7 | ⭐⭐ |
| Full Attention Pass | 8 | ⭐⭐⭐ |

In [ ]:
import numpy as np
import math

np.random.seed(42)
np.set_printoptions(precision=8, suppress=True)

try:
    import torch
    import torch.nn.functional as F
    torch.manual_seed(42)
    HAS_TORCH = True
    print(f'NumPy {np.__version__} | PyTorch {torch.__version__} — Ready!')
except ImportError:
    HAS_TORCH = False
    print(f'NumPy {np.__version__} — Ready! (PyTorch exercises will use NumPy fallbacks)')

---

## Exercise 1: Index Identification ⭐

The most fundamental skill in index notation is recognising **free indices** (appear once, define output shape)
vs **dummy indices** (appear twice, summed over).

**Task**: For each expression below, identify:
1. All **free indices**
2. All **dummy indices**
3. The **resulting shape** (scalar / vector / matrix)
4. The equivalent **`np.einsum`** string

**Expressions**:

| # | Expression | Free | Dummy | Shape | Einsum |
|---|-----------|------|-------|-------|--------|
| a | $c_i = A_{ij} b_j$ | ? | ? | ? | ? |
| b | $C_{ik} = A_{ij} B_{jk}$ | ? | ? | ? | ? |
| c | $s = v_i v_i$ | ? | ? | ? | ? |
| d | $T_{ij} = u_i v_j$ | ? | ? | ? | ? |
| e | $D_{il} = A_{ij} B_{jk} C_{kl}$ | ? | ? | ? | ? |

In [ ]:
# Exercise 1: Your Solution

def identify_indices():
    """
    Return a list of dicts, one per expression, with keys:
    'free', 'dummy', 'shape', 'einsum'
    """
    results = [
        # Expression a: c_i = A_ij b_j
        {'free': None, 'dummy': None, 'shape': None, 'einsum': None},
        # Expression b: C_ik = A_ij B_jk
        {'free': None, 'dummy': None, 'shape': None, 'einsum': None},
        # Expression c: s = v_i v_i
        {'free': None, 'dummy': None, 'shape': None, 'einsum': None},
        # Expression d: T_ij = u_i v_j
        {'free': None, 'dummy': None, 'shape': None, 'einsum': None},
        # Expression e: D_il = A_ij B_jk C_kl
        {'free': None, 'dummy': None, 'shape': None, 'einsum': None},
    ]
    # YOUR CODE HERE
    return results

# Verify with actual computation
n, m, p, q = 4, 5, 6, 3
A = np.random.randn(n, m)
b = np.random.randn(m)
B = np.random.randn(m, p)
C_mat = np.random.randn(p, q)
u = np.random.randn(n)
v = np.random.randn(m)

# Print shapes of results to verify
print(f'(a) A @ b shape: {(A @ b).shape}')
print(f'(b) A @ B shape: {(A @ B).shape}')
print(f'(c) v . v shape: scalar = {np.dot(v, v):.4f}')
print(f'(d) outer(u, v) shape: {np.outer(u, v).shape}')
print(f'(e) A @ B @ C shape: {(A @ B @ C_mat).shape}')

In [ ]:
# Exercise 1: Solution ✅

def identify_indices_solution():
    results = [
        # (a) c_i = A_ij b_j  →  i appears once (free), j appears twice (dummy)
        {'free': ['i'], 'dummy': ['j'], 'shape': 'vector (n,)', 'einsum': 'ij,j->i'},
        # (b) C_ik = A_ij B_jk  →  i,k free; j dummy
        {'free': ['i', 'k'], 'dummy': ['j'], 'shape': 'matrix (n,p)', 'einsum': 'ij,jk->ik'},
        # (c) s = v_i v_i  →  no free; i dummy
        {'free': [], 'dummy': ['i'], 'shape': 'scalar', 'einsum': 'i,i->'},
        # (d) T_ij = u_i v_j  →  i,j free; no dummy (outer product)
        {'free': ['i', 'j'], 'dummy': [], 'shape': 'matrix (n,m)', 'einsum': 'i,j->ij'},
        # (e) D_il = A_ij B_jk C_kl  →  i,l free; j,k dummy
        {'free': ['i', 'l'], 'dummy': ['j', 'k'], 'shape': 'matrix (n,q)', 'einsum': 'ij,jk,kl->il'},
    ]
    return results

# Verify each einsum matches the standard operation
n, m, p, q = 4, 5, 6, 3
A = np.random.randn(n, m)
b = np.random.randn(m)
B = np.random.randn(m, p)
C_mat = np.random.randn(p, q)
u = np.random.randn(n)
v = np.random.randn(m)

sol = identify_indices_solution()

# (a) Matrix-vector product
result_a = np.einsum(sol[0]['einsum'], A, b)
assert np.allclose(result_a, A @ b)
print(f'(a) ij,j->i  ✓  free={sol[0]["free"]}, dummy={sol[0]["dummy"]}, shape={result_a.shape}')

# (b) Matrix multiplication
result_b = np.einsum(sol[1]['einsum'], A, B)
assert np.allclose(result_b, A @ B)
print(f'(b) ij,jk->ik  ✓  free={sol[1]["free"]}, dummy={sol[1]["dummy"]}, shape={result_b.shape}')

# (c) Dot product (scalar)
result_c = np.einsum(sol[2]['einsum'], v, v)
assert np.isclose(result_c, np.dot(v, v))
print(f'(c) i,i->  ✓  free={sol[2]["free"]}, dummy={sol[2]["dummy"]}, result={result_c:.6f}')

# (d) Outer product
result_d = np.einsum(sol[3]['einsum'], u, v)
assert np.allclose(result_d, np.outer(u, v))
print(f'(d) i,j->ij  ✓  free={sol[3]["free"]}, dummy={sol[3]["dummy"]}, shape={result_d.shape}')

# (e) Triple matrix product
result_e = np.einsum(sol[4]['einsum'], A, B, C_mat)
assert np.allclose(result_e, A @ B @ C_mat)
print(f'(e) ij,jk,kl->il  ✓  free={sol[4]["free"]}, dummy={sol[4]["dummy"]}, shape={result_e.shape}')

print('\nKey insight: free indices define output dimensions, dummy indices are summed away.')
print('The einsum string is a direct transcription of the index expression.')

---

## Exercise 2: Kronecker Delta Manipulation ⭐

The Kronecker delta $\delta_{ij}$ acts as an **index substitution operator**: $\delta_{ij} x_j = x_i$.
This is the most important simplification rule in index notation.

**Task**: Simplify each expression analytically, then verify numerically.

1. $\delta_{ij} \delta_{jk}$
2. $\delta_{ij} A_{jk} \delta_{kl}$
3. $\delta_{ii}$ where $i = 1, \dots, n$
4. $\varepsilon_{ijk} \delta_{jk}$

**Hint**: For each, apply the substitution property step by step. For (4), think about contracting symmetric with antisymmetric tensors.

In [ ]:
# Exercise 2: Your Solution

n = 4  # dimension for testing

# Build Kronecker delta and Levi-Civita symbol
delta = np.eye(n)

# Levi-Civita symbol (3D only)
def levi_civita_3d():
    """Return the 3x3x3 Levi-Civita tensor."""
    eps = np.zeros((3, 3, 3))
    # YOUR CODE HERE
    pass
    return eps

# Simplification 1: delta_ij delta_jk = ?
# YOUR CODE HERE
result_1 = None

# Simplification 2: delta_ij A_jk delta_kl = ?
A = np.random.randn(n, n)
# YOUR CODE HERE
result_2 = None

# Simplification 3: delta_ii = ?
# YOUR CODE HERE
result_3 = None

# Simplification 4: eps_ijk delta_jk = ?
# YOUR CODE HERE
result_4 = None

In [ ]:
# Exercise 2: Solution ✅

n = 4
delta = np.eye(n)

# --- Simplification 1: δ_ij δ_jk = δ_ik ---
# Contract over j: first delta forces j=i, giving δ_ik
result_1 = np.einsum('ij,jk->ik', delta, delta)
assert np.allclose(result_1, delta)
print('1. δ_ij δ_jk = δ_ik  ✓')
print(f'   Result is identity matrix: {np.allclose(result_1, np.eye(n))}')

# --- Simplification 2: δ_ij A_jk δ_kl = A_il ---
# First delta: j → i, giving A_ik δ_kl. Second delta: k → l, giving A_il.
A = np.random.randn(n, n)
result_2 = np.einsum('ij,jk,kl->il', delta, A, delta)
assert np.allclose(result_2, A)
print('\n2. δ_ij A_jk δ_kl = A_il  ✓')
print(f'   Sandwiching with deltas leaves A unchanged: {np.allclose(result_2, A)}')

# --- Simplification 3: δ_ii = n (trace of identity) ---
result_3 = np.einsum('ii->', delta)
assert result_3 == n
print(f'\n3. δ_ii = {result_3:.0f}  (= n = {n})  ✓')
print(f'   This is tr(I_n) = n')

# --- Simplification 4: ε_ijk δ_jk = 0 ---
# δ_jk is symmetric; ε_ijk is antisymmetric in (j,k). Contraction = 0.
eps = np.zeros((3, 3, 3))
for i in range(3):
    for j in range(3):
        for k in range(3):
            eps[i, j, k] = (
                (i - j) * (j - k) * (k - i) / 2
                if len({i, j, k}) == 3 else 0
            )

delta3 = np.eye(3)
result_4 = np.einsum('ijk,jk->i', eps, delta3)
assert np.allclose(result_4, 0)
print(f'\n4. ε_ijk δ_jk = {result_4}  ✓')
print('   Symmetric ⊗ antisymmetric = 0 (always!)')

# Bonus: verify Levi-Civita is correct by computing a cross product
a = np.array([1.0, 2.0, 3.0])
b_vec = np.array([4.0, 5.0, 6.0])
cross_einsum = np.einsum('ijk,j,k->i', eps, a, b_vec)
cross_numpy = np.cross(a, b_vec)
assert np.allclose(cross_einsum, cross_numpy)
print(f'\nBonus: ε_ijk a_j b_k = cross(a,b) = {cross_einsum}  ✓')

---

## Exercise 3: Gradient Derivation ⭐⭐

Index notation turns gradient derivation into a **mechanical** process: apply the product rule, use
$\frac{\partial x_i}{\partial x_j} = \delta_{ij}$, and simplify with the Kronecker delta.

**Task**: Derive the gradient for each scalar loss below using index notation. Show all steps.

### 3a. Quadratic Form

$L = x_i A_{ij} x_j$

Find $\frac{\partial L}{\partial x_k}$.

### 3b. Frobenius Norm Loss

$L = \| Y - X W \|_F^2$ where $Y_{ij}, X_{ik}, W_{kj}$ are matrices.

Find $\frac{\partial L}{\partial W_{pq}}$.

### 3c. Cross-Entropy + Softmax

$L = -y_j \log(\text{softmax}(z)_j)$ where $y$ is one-hot.

Find $\frac{\partial L}{\partial z_m}$.

In [ ]:
# Exercise 3: Your Solution

# --- 3a: Quadratic form L = x_i A_ij x_j ---
def grad_quadratic(A, x):
    """
    Compute dL/dx for L = x^T A x using index notation.
    
    Derivation steps:
    dL/dx_k = d/dx_k (x_i A_ij x_j)
            = δ_ik A_ij x_j + x_i A_ij δ_jk    (product rule)
            = ?
    """
    # YOUR CODE HERE
    pass

# --- 3b: Frobenius norm L = ||Y - XW||_F^2 ---
def grad_frobenius(X, Y, W):
    """
    Compute dL/dW for L = ||Y - XW||_F^2 using index notation.
    
    Let R_ij = Y_ij - X_ik W_kj (residual).
    L = R_ij R_ij
    
    dL/dW_pq = 2 R_ij * dR_ij/dW_pq
             = 2 R_ij * (-X_ip δ_jq)
             = ?
    """
    # YOUR CODE HERE
    pass

# --- 3c: Cross-entropy + softmax ---
def grad_cross_entropy_softmax(z, y):
    """
    Compute dL/dz for L = -y_j log(softmax(z)_j).
    
    Let p_i = softmax(z)_i.
    dL/dz_m = -y_j (1/p_j) * dp_j/dz_m
            = -y_j (1/p_j) * p_j(δ_jm - p_m)
            = ?
    """
    # YOUR CODE HERE
    pass

# Test
n_test = 5
A_test = np.random.randn(n_test, n_test)
x_test = np.random.randn(n_test)

print("3a gradient:", grad_quadratic(A_test, x_test))
print("Expected:   ", (A_test + A_test.T) @ x_test)

In [ ]:
# Exercise 3: Solution ✅

# ═══════════════════════════════════════════════════════════════
# 3a: Quadratic Form  L = x_i A_ij x_j
# ═══════════════════════════════════════════════════════════════
#
# dL/dx_k = d/dx_k (x_i A_ij x_j)
#         = δ_ik A_ij x_j  +  x_i A_ij δ_jk      (product rule)
#         = A_kj x_j       +  x_i A_ik             (delta substitution)
#         = A_kj x_j       +  A_jk x_j             (rename i→j in 2nd term)
#         = (A_kj + A_jk) x_j
#
# In matrix form: ∇_x L = (A + A^T) x
# If A is symmetric: ∇_x L = 2Ax

def grad_quadratic_sol(A, x):
    return (A + A.T) @ x

n_test = 5
A_test = np.random.randn(n_test, n_test)
x_test = np.random.randn(n_test)

# Verify with finite differences
grad_analytical = grad_quadratic_sol(A_test, x_test)
eps = 1e-7
grad_numerical = np.zeros(n_test)
for k in range(n_test):
    x_plus = x_test.copy(); x_plus[k] += eps
    x_minus = x_test.copy(); x_minus[k] -= eps
    L_plus = x_plus @ A_test @ x_plus
    L_minus = x_minus @ A_test @ x_minus
    grad_numerical[k] = (L_plus - L_minus) / (2 * eps)

assert np.allclose(grad_analytical, grad_numerical, atol=1e-5)
print("3a. ∇_x (x^T A x) = (A + A^T)x  ✓")
print(f"    Max error vs finite diff: {np.max(np.abs(grad_analytical - grad_numerical)):.2e}")

# Verify einsum version
grad_einsum = np.einsum('kj,j->k', A_test, x_test) + np.einsum('jk,j->k', A_test, x_test)
assert np.allclose(grad_einsum, grad_analytical)
print("    Einsum verification: ✓")

# ═══════════════════════════════════════════════════════════════
# 3b: Frobenius Norm  L = ||Y - XW||_F^2
# ═══════════════════════════════════════════════════════════════
#
# R_ij = Y_ij - X_ik W_kj    (residual)
# L = R_ij R_ij
#
# dL/dW_pq = 2 R_ij · (dR_ij / dW_pq)
#          = 2 R_ij · (-X_ip δ_jq)       (only W_kj depends on W_pq: k→p, j→q)
#          = -2 X_ip R_iq                  (delta kills j, sets j=q)
#
# Matrix form: dL/dW = -2 X^T (Y - XW)

def grad_frobenius_sol(X, Y, W):
    R = Y - X @ W
    return -2 * X.T @ R

m, n_feat, p = 10, 5, 3
X_test = np.random.randn(m, n_feat)
Y_test = np.random.randn(m, p)
W_test = np.random.randn(n_feat, p)

grad_W = grad_frobenius_sol(X_test, Y_test, W_test)

# Verify with finite differences
grad_W_num = np.zeros_like(W_test)
for pp in range(n_feat):
    for qq in range(p):
        W_plus = W_test.copy(); W_plus[pp, qq] += eps
        W_minus = W_test.copy(); W_minus[pp, qq] -= eps
        L_plus = np.sum((Y_test - X_test @ W_plus) ** 2)
        L_minus = np.sum((Y_test - X_test @ W_minus) ** 2)
        grad_W_num[pp, qq] = (L_plus - L_minus) / (2 * eps)

assert np.allclose(grad_W, grad_W_num, atol=1e-5)
print("\n3b. ∂L/∂W = -2 X^T(Y - XW)  ✓")
print(f"    Max error vs finite diff: {np.max(np.abs(grad_W - grad_W_num)):.2e}")

# Show einsum equivalent
R_test = Y_test - X_test @ W_test
grad_W_einsum = -2 * np.einsum('ip,iq->pq', X_test, R_test)
assert np.allclose(grad_W_einsum, grad_W)
print("    Einsum 'ip,iq->pq' verification: ✓")

# ═══════════════════════════════════════════════════════════════
# 3c: Cross-Entropy + Softmax
# ═══════════════════════════════════════════════════════════════
#
# p_i = softmax(z)_i = exp(z_i) / Σ_k exp(z_k)
# L = -y_j log(p_j)
#
# dL/dz_m = -y_j · (1/p_j) · dp_j/dz_m
#         = -y_j · (1/p_j) · p_j(δ_jm - p_m)    (softmax Jacobian)
#         = -y_j (δ_jm - p_m)
#         = -y_m + p_m Σ_j y_j
#         = -y_m + p_m · 1                          (y is one-hot: Σy = 1)
#         = p_m - y_m
#
# Vector form: ∇_z L = softmax(z) - y

def softmax(z):
    e = np.exp(z - z.max())  # subtract max for numerical stability
    return e / e.sum()

def grad_ce_softmax_sol(z, y):
    return softmax(z) - y

z_test = np.random.randn(5)
y_test = np.zeros(5); y_test[2] = 1.0  # one-hot at class 2

grad_z = grad_ce_softmax_sol(z_test, y_test)

# Verify with finite differences
grad_z_num = np.zeros(5)
for mm in range(5):
    z_plus = z_test.copy(); z_plus[mm] += eps
    z_minus = z_test.copy(); z_minus[mm] -= eps
    p_plus = softmax(z_plus)
    p_minus = softmax(z_minus)
    L_plus = -y_test @ np.log(p_plus + 1e-12)
    L_minus = -y_test @ np.log(p_minus + 1e-12)
    grad_z_num[mm] = (L_plus - L_minus) / (2 * eps)

assert np.allclose(grad_z, grad_z_num, atol=1e-5)
print("\n3c. ∇_z L(cross-entropy+softmax) = softmax(z) - y  ✓")
print(f"    Max error vs finite diff: {np.max(np.abs(grad_z - grad_z_num)):.2e}")
print(f"    Gradient: {grad_z}")
print(f"    softmax:  {softmax(z_test)}")
print(f"    y:        {y_test}")
print("\nKey insight: the gradient is simply (prediction - target).")
print("This is why cross-entropy + softmax is universally used in classification.")

---

## Exercise 4: Einsum Implementation ⭐⭐

`np.einsum` translates index notation directly into executable code. The string format mirrors
the mathematical expression: `'ij,jk->ik'` means $C_{ik} = A_{ij} B_{jk}$.

**Task**: Write the einsum string and verify against standard NumPy for each operation:

### 4a. Batched Trace

Given $A$ with shape $(B, n, n)$, compute the trace of each matrix: $t_b = A_{bii}$.

### 4b. Bilinear Form

Given $x$ shape $(n,)$, $M$ shape $(n, n)$, $y$ shape $(n,)$, compute $s = x_i M_{ij} y_j$.

### 4c. Multi-Head Attention Scores

Given $Q$ shape $(B, H, T, d_k)$, $K$ shape $(B, H, T, d_k)$, compute $S_{bhij} = Q_{bhid} K_{bhjd}$.

In [ ]:
# Exercise 4: Your Solution

B, n, H, T, d_k = 3, 5, 4, 8, 6

# 4a: Batched trace — A shape (B, n, n)
A_batch = np.random.randn(B, n, n)

# t_b = A_bii → einsum string = ?
# YOUR CODE HERE
traces = None  # np.einsum('???', A_batch)
print(f'4a traces: {traces}')

# 4b: Bilinear form — x (n,), M (n,n), y (n,)
x_vec = np.random.randn(n)
M_mat = np.random.randn(n, n)
y_vec = np.random.randn(n)

# s = x_i M_ij y_j → einsum string = ?
# YOUR CODE HERE
s_bilinear = None  # np.einsum('???', x_vec, M_mat, y_vec)
print(f'4b bilinear form: {s_bilinear}')

# 4c: Multi-head attention scores — Q,K shape (B, H, T, d_k)
Q = np.random.randn(B, H, T, d_k)
K = np.random.randn(B, H, T, d_k)

# S_bhij = Q_bhid K_bhjd → einsum string = ?
# YOUR CODE HERE
S_scores = None  # np.einsum('???', Q, K)
print(f'4c attention scores shape: {S_scores.shape if S_scores is not None else None}')

In [ ]:
# Exercise 4: Solution ✅

B, n, H, T, d_k = 3, 5, 4, 8, 6

# ═══════════════════════════════════════════════════════════════
# 4a: Batched Trace   t_b = A_bii
# ═══════════════════════════════════════════════════════════════
A_batch = np.random.randn(B, n, n)

# Index notation: t_b = A_bii  →  'bii->b'
# i is dummy (appears twice), b is free
traces = np.einsum('bii->b', A_batch)

# Verify against np.trace
traces_ref = np.trace(A_batch, axis1=1, axis2=2)
assert np.allclose(traces, traces_ref)
print("4a. Batched trace: 'bii->b'  ✓")
print(f"    traces = {traces}")
print(f"    shapes: ({B},{n},{n}) → ({B},)")

# ═══════════════════════════════════════════════════════════════
# 4b: Bilinear Form   s = x_i M_ij y_j
# ═══════════════════════════════════════════════════════════════
x_vec = np.random.randn(n)
M_mat = np.random.randn(n, n)
y_vec = np.random.randn(n)

# Index notation: s = x_i M_ij y_j  →  'i,ij,j->'
# Both i and j are dummy, no free indices → scalar output
s_bilinear = np.einsum('i,ij,j->', x_vec, M_mat, y_vec)

# Verify: x^T M y = (x @ M) @ y
s_ref = x_vec @ M_mat @ y_vec
assert np.isclose(s_bilinear, s_ref)
print(f"\n4b. Bilinear form: 'i,ij,j->'  ✓")
print(f"    s = {s_bilinear:.6f}")
print(f"    shapes: ({n},) × ({n},{n}) × ({n},) → scalar")

# ═══════════════════════════════════════════════════════════════
# 4c: Multi-Head Attention Scores   S_bhij = Q_bhid K_bhjd
# ═══════════════════════════════════════════════════════════════
Q = np.random.randn(B, H, T, d_k)
K = np.random.randn(B, H, T, d_k)

# Index notation: S_bhij = Q_bhid K_bhjd  →  'bhid,bhjd->bhij'
# d is dummy (key dimension summed), b,h,i,j are free
S_scores = np.einsum('bhid,bhjd->bhij', Q, K)

# Verify: equivalent to Q @ K^T along last two dims
S_ref = Q @ K.transpose(0, 1, 3, 2)  # (B,H,T,d_k) @ (B,H,d_k,T) = (B,H,T,T)
assert np.allclose(S_scores, S_ref)
print(f"\n4c. Attention scores: 'bhid,bhjd->bhij'  ✓")
print(f"    shapes: ({B},{H},{T},{d_k}) × ({B},{H},{T},{d_k}) → {S_scores.shape}")
print(f"    This is Q K^T for each (batch, head) pair.")

# Bonus: other common attention einsums
V = np.random.randn(B, H, T, d_k)
alpha = np.random.randn(B, H, T, T)
alpha = np.exp(alpha) / np.exp(alpha).sum(axis=-1, keepdims=True)  # fake softmax

# Output: O_bhia = alpha_bhij V_bhja  →  'bhij,bhja->bhia'
O = np.einsum('bhij,bhja->bhia', alpha, V)
O_ref = alpha @ V
assert np.allclose(O, O_ref)
print(f"\n    Bonus: attention output 'bhij,bhja->bhia' → {O.shape}  ✓")

---

## Exercise 5: Contraction Order Optimisation ⭐⭐

The order in which you contract indices can change computational cost by **orders of magnitude**.
This is directly relevant to LoRA and other low-rank methods.

**Task**: Consider $C_{il} = A_{ij} B_{jk} D_{kl}$ where:
- $A$ is $(10000, 512)$  — e.g., activations
- $B$ is $(512, 4)$      — e.g., LoRA down-projection
- $D$ is $(4, 512)$      — e.g., LoRA up-projection

1. Compute the total FLOPs for **left-to-right** evaluation: $(A_{ij} B_{jk}) D_{kl}$
2. Compute the total FLOPs for **right-to-left** evaluation: $A_{ij} (B_{jk} D_{kl})$
3. Which order is faster, and by how much?
4. Verify by timing both orders with `np.einsum`

**Hint**: For matrix multiply $(m,n) \times (n,p)$, FLOPs $\approx m \times p \times n$ (multiply-adds).

In [ ]:
# Exercise 5: Your Solution

# Dimensions (LoRA-style)
n_samples = 10000
d_model = 512
r = 4  # LoRA rank

A = np.random.randn(n_samples, d_model)
B = np.random.randn(d_model, r)
D = np.random.randn(r, d_model)

# 5a: Left-to-right FLOPs
# Step 1: T_ik = A_ij B_jk  →  (10000, 512) × (512, 4) = (10000, 4)
#   FLOPs = ?
# Step 2: C_il = T_ik D_kl  →  (10000, 4) × (4, 512) = (10000, 512)
#   FLOPs = ?
# Total = ?

flops_left_to_right = None  # YOUR CODE HERE

# 5b: Right-to-left FLOPs
# Step 1: E_jl = B_jk D_kl  →  (512, 4) × (4, 512) = (512, 512)
#   FLOPs = ?
# Step 2: C_il = A_ij E_jl  →  (10000, 512) × (512, 512) = (10000, 512)
#   FLOPs = ?
# Total = ?

flops_right_to_left = None  # YOUR CODE HERE

# 5c: Speedup ratio
speedup = None  # YOUR CODE HERE

print(f"Left-to-right FLOPs:  {flops_left_to_right}")
print(f"Right-to-left FLOPs:  {flops_right_to_left}")
print(f"Speedup ratio:        {speedup}x")

In [ ]:
# Exercise 5: Solution ✅

import time

n_samples = 10000
d_model = 512
r = 4

A = np.random.randn(n_samples, d_model)
B = np.random.randn(d_model, r)
D = np.random.randn(r, d_model)

# ═══════════════════════════════════════════════════════════════
# 5a: Left-to-right:  (A B) D
# ═══════════════════════════════════════════════════════════════
# Step 1: T = A @ B → (10000, 512) × (512, 4) = (10000, 4)
#   FLOPs = 10000 × 4 × 512 = 20,480,000
flops_step1_LR = n_samples * r * d_model

# Step 2: C = T @ D → (10000, 4) × (4, 512) = (10000, 512)
#   FLOPs = 10000 × 512 × 4 = 20,480,000
flops_step2_LR = n_samples * d_model * r

flops_LR = flops_step1_LR + flops_step2_LR
print(f"Left-to-right:")
print(f"  Step 1: {n_samples}×{r}×{d_model} = {flops_step1_LR:,}")
print(f"  Step 2: {n_samples}×{d_model}×{r} = {flops_step2_LR:,}")
print(f"  Total:  {flops_LR:,} FLOPs")

# ═══════════════════════════════════════════════════════════════
# 5b: Right-to-left:  A (B D)
# ═══════════════════════════════════════════════════════════════
# Step 1: E = B @ D → (512, 4) × (4, 512) = (512, 512)
#   FLOPs = 512 × 512 × 4 = 1,048,576
flops_step1_RL = d_model * d_model * r

# Step 2: C = A @ E → (10000, 512) × (512, 512) = (10000, 512)
#   FLOPs = 10000 × 512 × 512 = 2,621,440,000
flops_step2_RL = n_samples * d_model * d_model

flops_RL = flops_step1_RL + flops_step2_RL
print(f"\nRight-to-left:")
print(f"  Step 1: {d_model}×{d_model}×{r} = {flops_step1_RL:,}")
print(f"  Step 2: {n_samples}×{d_model}×{d_model} = {flops_step2_RL:,}")
print(f"  Total:  {flops_RL:,} FLOPs")

# ═══════════════════════════════════════════════════════════════
# 5c: Speedup ratio
# ═══════════════════════════════════════════════════════════════
speedup = flops_RL / flops_LR
print(f"\nSpeedup: {speedup:.1f}×  (left-to-right is faster)")

# 5d: Timing verification
n_runs = 5

t_start = time.time()
for _ in range(n_runs):
    C_LR = (A @ B) @ D
t_LR = (time.time() - t_start) / n_runs

t_start = time.time()
for _ in range(n_runs):
    C_RL = A @ (B @ D)
t_RL = (time.time() - t_start) / n_runs

assert np.allclose(C_LR, C_RL)  # same result, different cost
print(f"\nTiming (avg of {n_runs} runs):")
print(f"  Left-to-right:  {t_LR*1000:.2f} ms")
print(f"  Right-to-left:  {t_RL*1000:.2f} ms")
print(f"  Measured ratio: {t_RL/t_LR:.1f}×")

# einsum with optimize=True automatically finds the best order
t_start = time.time()
for _ in range(n_runs):
    C_opt = np.einsum('ij,jk,kl->il', A, B, D, optimize=True)
t_opt = (time.time() - t_start) / n_runs
print(f"  einsum(optimize): {t_opt*1000:.2f} ms")

print("\nKey insight: In LoRA, always multiply through the low-rank bottleneck first.")
print("This is why LoRA adds negligible overhead despite modifying large weight matrices.")

---

## Exercise 6: Softmax Gradient Verification ⭐⭐

The softmax Jacobian is one of the most important results in deep learning. Deriving it in index
notation demonstrates the power of the $\delta_{ij}$ substitution rule.

**Task**:

**6a.** Derive the Jacobian $\frac{\partial p_i}{\partial z_j}$ where $p_i = \text{softmax}(z)_i = \frac{e^{z_i}}{\sum_k e^{z_k}}$
using the quotient rule in index notation. Show the result involves $\delta_{ij}$.

**6b.** Implement the analytical Jacobian and verify it against central finite differences.

**6c.** Show that $J = \text{diag}(p) - p p^T$ and verify this matrix form.

In [ ]:
# Exercise 6: Your Solution

def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()

def softmax_jacobian(z):
    """
    Compute the Jacobian J_ij = dp_i/dz_j using index notation.
    
    Derivation:
    p_i = exp(z_i) / S,  where S = Σ_k exp(z_k)
    
    dp_i/dz_j = [δ_ij exp(z_i) S - exp(z_i) exp(z_j)] / S²
               = (exp(z_i)/S) [δ_ij - exp(z_j)/S]
               = p_i (δ_ij - p_j)
    
    Matrix form: J = diag(p) - p p^T
    """
    n = len(z)
    # YOUR CODE HERE
    J = np.zeros((n, n))
    pass
    return J

z = np.random.randn(5)

# 6b: Verify against finite differences
J_analytical = softmax_jacobian(z)
# YOUR CODE HERE: compute J_numerical with central finite differences

# 6c: Verify matrix form
# YOUR CODE HERE: compute J_matrix = diag(p) - outer(p, p)

In [ ]:
# Exercise 6: Solution ✅

def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()

# ═══════════════════════════════════════════════════════════════
# 6a: Derive the Jacobian in index notation
# ═══════════════════════════════════════════════════════════════
#
# p_i = exp(z_i) / S,   S = Σ_k exp(z_k)
#
# Apply quotient rule:
#   dp_i/dz_j = [δ_ij · exp(z_i) · S  -  exp(z_i) · exp(z_j)] / S²
#             = exp(z_i)/S · [δ_ij - exp(z_j)/S]
#             = p_i · (δ_ij - p_j)
#
# In matrix form:
#   J_ij = p_i δ_ij - p_i p_j
#        = diag(p) - p p^T

def softmax_jacobian_sol(z):
    """J_ij = p_i (δ_ij − p_j) = diag(p) − p p^T"""
    p = softmax(z)
    # Method 1: direct from index notation
    n = len(z)
    J = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            J[i, j] = p[i] * ((1 if i == j else 0) - p[j])
    return J

z = np.random.randn(5)
p = softmax(z)
J_analytical = softmax_jacobian_sol(z)

print("6a. Softmax Jacobian: J_ij = p_i(δ_ij − p_j)")
print(f"    z = {z}")
print(f"    p = softmax(z) = {p}")

# ═══════════════════════════════════════════════════════════════
# 6b: Verify against finite differences
# ═══════════════════════════════════════════════════════════════
eps = 1e-7
n = len(z)
J_numerical = np.zeros((n, n))
for j in range(n):
    z_plus = z.copy();  z_plus[j] += eps
    z_minus = z.copy(); z_minus[j] -= eps
    J_numerical[:, j] = (softmax(z_plus) - softmax(z_minus)) / (2 * eps)

max_err = np.max(np.abs(J_analytical - J_numerical))
assert max_err < 1e-6
print(f"\n6b. Finite difference verification  ✓")
print(f"    Max error: {max_err:.2e}")

# ═══════════════════════════════════════════════════════════════
# 6c: Matrix form  J = diag(p) − p p^T
# ═══════════════════════════════════════════════════════════════
J_matrix = np.diag(p) - np.outer(p, p)
assert np.allclose(J_analytical, J_matrix)
print(f"\n6c. Matrix form: diag(p) − pp^T  ✓")
print(f"    ||J_loop − J_matrix||_max = {np.max(np.abs(J_analytical - J_matrix)):.2e}")

# Einsum version of J = diag(p) - p p^T
# diag(p)_ij = p_i δ_ij  →  can't directly einsum δ, but outer product is:
J_outer = np.einsum('i,j->ij', p, p)  # p p^T
J_einsum = np.diag(p) - J_outer
assert np.allclose(J_einsum, J_matrix)
print(f"    Einsum 'i,j->ij' for outer product  ✓")

# Key properties of softmax Jacobian
print(f"\nProperties:")
print(f"    Rows sum to zero: {np.allclose(J_matrix.sum(axis=1), 0)}  (dp/dz is zero-sum)")
print(f"    Symmetric: {np.allclose(J_matrix, J_matrix.T)}  (J = J^T)")
print(f"    Diagonal elements p_i(1-p_i) > 0: {np.all(np.diag(J_matrix) > 0)}")
print(f"    Off-diagonal elements -p_i p_j < 0: {np.all(J_matrix - np.diag(np.diag(J_matrix)) <= 0)}")

---

## Exercise 7: Symmetry Analysis ⭐⭐

Symmetry and antisymmetry are **structural properties** visible directly in index notation.
A tensor $T_{ij}$ is symmetric if $T_{ij} = T_{ji}$, antisymmetric if $T_{ij} = -T_{ji}$.

**Task**: For each tensor expression, determine if it is **symmetric**, **antisymmetric**, or **neither**
in the indicated pair of indices. Prove your answer analytically AND verify numerically.

| # | Expression | Indices | Type? |
|---|-----------|---------|-------|
| 1 | $S_{ij} = A_{ik} A_{jk}$ (i.e., $AA^T$) | $(i, j)$ | ? |
| 2 | $T_{ij} = A_{ik} B_{kj} - A_{jk} B_{ki}$ | $(i, j)$ | ? |
| 3 | $H_{ij} = \frac{\partial^2 f}{\partial x_i \partial x_j}$ (Hessian) | $(i, j)$ | ? |

**Hint**: Swap $i \leftrightarrow j$ and see if you get the same expression, or its negation.

In [ ]:
# Exercise 7: Your Solution

n = 5
A = np.random.randn(n, n)
B = np.random.randn(n, n)

# Expression 1: S_ij = A_ik A_jk  (= AA^T)
# Swap i↔j: S_ji = A_jk A_ik = A_ik A_jk = S_ij  →  ?
S = np.einsum('ik,jk->ij', A, A)  # AA^T
# YOUR ANALYSIS HERE
print(f"1. S_ij = A_ik A_jk")
print(f"   Is S symmetric?  {None}")  # Replace None
print(f"   Is S antisymmetric?  {None}")

# Expression 2: T_ij = A_ik B_kj - A_jk B_ki
# Swap i↔j: T_ji = A_jk B_ki - A_ik B_kj = -(A_ik B_kj - A_jk B_ki) = ?
T = np.einsum('ik,kj->ij', A, B) - np.einsum('jk,ki->ij', A, B)
# YOUR ANALYSIS HERE
print(f"\n2. T_ij = A_ik B_kj - A_jk B_ki")
print(f"   Is T symmetric?  {None}")
print(f"   Is T antisymmetric?  {None}")

# Expression 3: Hessian H_ij = d²f / dx_i dx_j
# For a C² function, Schwarz's theorem: d²f/dx_i dx_j = d²f/dx_j dx_i → ?
def compute_hessian(f, x, eps=1e-5):
    """Numerical Hessian via finite differences."""
    n = len(x)
    H = np.zeros((n, n))
    # YOUR CODE HERE
    pass
    return H

f = lambda x: np.sum(x**3) + x[0]*x[1]**2 + np.sin(x[0]*x[2])
x_test = np.random.randn(4)
# H = compute_hessian(f, x_test)
print(f"\n3. Hessian H_ij")
print(f"   Is H symmetric?  {None}")

In [ ]:
# Exercise 7: Solution ✅

n = 5
A = np.random.randn(n, n)
B = np.random.randn(n, n)

# ═══════════════════════════════════════════════════════════════
# Expression 1: S_ij = A_ik A_jk  (= AA^T)  →  SYMMETRIC
# ═══════════════════════════════════════════════════════════════
#
# Proof: swap i ↔ j
#   S_ji = A_jk A_ik
#        = A_ik A_jk    (scalar multiplication is commutative)
#        = S_ij          ✓
#
# Alternatively: AA^T is always symmetric because (AA^T)^T = (A^T)^T A^T = AA^T

S = np.einsum('ik,jk->ij', A, A)
assert np.allclose(S, S.T), "S should be symmetric"
print("1. S_ij = A_ik A_jk (= AA^T): SYMMETRIC  ✓")
print(f"   ||S - S^T||_max = {np.max(np.abs(S - S.T)):.2e}")

# Also verify: AA^T is positive semi-definite (eigenvalues ≥ 0)
eigenvalues = np.linalg.eigvalsh(S)
print(f"   Eigenvalues ≥ 0: {np.all(eigenvalues >= -1e-10)}  (positive semi-definite)")

# ═══════════════════════════════════════════════════════════════
# Expression 2: T_ij = A_ik B_kj - A_jk B_ki  →  ANTISYMMETRIC
# ═══════════════════════════════════════════════════════════════
#
# Proof: swap i ↔ j
#   T_ji = A_jk B_ki - A_ik B_kj
#        = -(A_ik B_kj - A_jk B_ki)
#        = -T_ij          ✓
#
# This is AB - (AB)^T, which is always antisymmetric.

T = np.einsum('ik,kj->ij', A, B) - np.einsum('jk,ki->ij', A, B)
assert np.allclose(T, -T.T), "T should be antisymmetric"
print(f"\n2. T_ij = A_ik B_kj - A_jk B_ki: ANTISYMMETRIC  ✓")
print(f"   ||T + T^T||_max = {np.max(np.abs(T + T.T)):.2e}")

# Property: diagonal of antisymmetric matrix is always zero
print(f"   Diagonal = {np.diag(T)}")
print(f"   Diagonal ≈ 0: {np.allclose(np.diag(T), 0)}")

# Verify it's AB - (AB)^T
AB = A @ B
assert np.allclose(T, AB - AB.T)
print(f"   T = AB - (AB)^T: ✓")

# ═══════════════════════════════════════════════════════════════
# Expression 3: Hessian H_ij = ∂²f/∂x_i∂x_j  →  SYMMETRIC
# ═══════════════════════════════════════════════════════════════
#
# By Schwarz's theorem (for C² functions):
#   ∂²f/∂x_i∂x_j = ∂²f/∂x_j∂x_i
# So H_ij = H_ji always (when f has continuous second derivatives).

def compute_hessian_sol(f, x, eps=1e-5):
    """Numerical Hessian via central finite differences."""
    n = len(x)
    H = np.zeros((n, n))
    f0 = f(x)
    for i in range(n):
        for j in range(n):
            x_pp = x.copy(); x_pp[i] += eps; x_pp[j] += eps
            x_pm = x.copy(); x_pm[i] += eps; x_pm[j] -= eps
            x_mp = x.copy(); x_mp[i] -= eps; x_mp[j] += eps
            x_mm = x.copy(); x_mm[i] -= eps; x_mm[j] -= eps
            H[i, j] = (f(x_pp) - f(x_pm) - f(x_mp) + f(x_mm)) / (4 * eps * eps)
    return H

# Test function with mixed terms
f = lambda x: np.sum(x**3) + x[0] * x[1]**2 + np.sin(x[0] * x[2])
x_test = np.random.randn(4)

H = compute_hessian_sol(f, x_test)
assert np.allclose(H, H.T, atol=1e-4), "Hessian should be symmetric"
print(f"\n3. Hessian H_ij = ∂²f/∂x_i∂x_j: SYMMETRIC  ✓  (Schwarz's theorem)")
print(f"   ||H - H^T||_max = {np.max(np.abs(H - H.T)):.2e}")
print(f"   H =")
for row in H:
    print(f"   [{', '.join(f'{v:8.4f}' for v in row)}]")

# Summary
print("\n┌────────────────────────────────────┬────────────────┐")
print("│ Expression                         │ Type           │")
print("├────────────────────────────────────┼────────────────┤")
print("│ A_ik A_jk  (AA^T)                  │ Symmetric      │")
print("│ A_ik B_kj − A_jk B_ki  (AB−(AB)^T) │ Antisymmetric  │")
print("│ ∂²f/∂x_i∂x_j  (Hessian)           │ Symmetric      │")
print("└────────────────────────────────────┴────────────────┘")

---

## Exercise 8: Full Attention Forward and Backward Pass ⭐⭐⭐

This exercise ties everything together: you will implement **single-head self-attention** entirely
in index notation using `np.einsum`, then derive and implement the backward pass,
and verify with numerical gradient checking.

Given input $X$ with shape $(T, d)$ and weight matrices $W^Q, W^K, W^V$ each $(d, d_k)$:

**Forward pass** (write each step in index notation AND einsum):
1. $Q_{ia} = X_{id} W^Q_{da}$
2. $S_{ij} = Q_{ia} K_{ja} / \sqrt{d_k}$
3. $\alpha_{ij} = \text{softmax}_j(S_{ij})$
4. $O_{ia} = \alpha_{ij} V_{ja}$

**Backward pass** — given upstream gradient $\frac{\partial L}{\partial O_{ia}}$:
1. Compute $\frac{\partial L}{\partial V}$, $\frac{\partial L}{\partial \alpha}$
2. Backprop through softmax
3. Compute $\frac{\partial L}{\partial Q}$, $\frac{\partial L}{\partial K}$
4. Compute $\frac{\partial L}{\partial W^Q}$, $\frac{\partial L}{\partial W^K}$, $\frac{\partial L}{\partial W^V}$

**Verify**: Check all weight gradients against finite differences.

In [ ]:
# Exercise 8: Your Solution

T_seq, d, d_k = 6, 8, 4
X = np.random.randn(T_seq, d)
WQ = np.random.randn(d, d_k)
WK = np.random.randn(d, d_k)
WV = np.random.randn(d, d_k)

def softmax_rows(S):
    """Row-wise softmax (stable)."""
    S_max = S.max(axis=1, keepdims=True)
    e = np.exp(S - S_max)
    return e / e.sum(axis=1, keepdims=True)

# ─── FORWARD PASS ───

# Step 1: Q_ia = X_id WQ_da  →  einsum: ?
Q = None  # YOUR CODE HERE
K = None
V = None

# Step 2: S_ij = Q_ia K_ja / √d_k  →  einsum: ?
S = None  # YOUR CODE HERE

# Step 3: α_ij = softmax_j(S_ij)
alpha = None  # YOUR CODE HERE

# Step 4: O_ia = α_ij V_ja  →  einsum: ?
O = None  # YOUR CODE HERE

# ─── BACKWARD PASS ───

dO = np.random.randn(T_seq, d_k)  # upstream gradient

# Step 4 backward: dL/dV and dL/dα
dV = None     # YOUR CODE HERE
dalpha = None # YOUR CODE HERE

# Step 3 backward: softmax gradient
dS = None  # YOUR CODE HERE

# Step 2 backward: dL/dQ and dL/dK
dQ = None  # YOUR CODE HERE
dK = None  # YOUR CODE HERE

# Step 1 backward: dL/dWQ, dL/dWK, dL/dWV
dWQ = None  # YOUR CODE HERE
dWK = None
dWV = None

print(f"Forward: O shape = {O.shape if O is not None else None}")
print(f"Backward: dWQ shape = {dWQ.shape if dWQ is not None else None}")

In [ ]:
# Exercise 8: Solution ✅ — Forward Pass

np.random.seed(123)  # fixed seed for reproducibility
T_seq, d, d_k = 6, 8, 4
X = np.random.randn(T_seq, d)
WQ = np.random.randn(d, d_k)
WK = np.random.randn(d, d_k)
WV = np.random.randn(d, d_k)

def softmax_rows(S):
    """Row-wise softmax (numerically stable)."""
    S_max = S.max(axis=1, keepdims=True)
    e = np.exp(S - S_max)
    return e / e.sum(axis=1, keepdims=True)

print("═══ FORWARD PASS ═══\n")

# ─────────────────────────────────────────────────────────────
# Step 1: Projections
#   Q_ia = X_id WQ_da    →    einsum: 'id,da->ia'
#   K_ia = X_id WK_da    →    einsum: 'id,da->ia'
#   V_ia = X_id WV_da    →    einsum: 'id,da->ia'
# ─────────────────────────────────────────────────────────────
Q = np.einsum('id,da->ia', X, WQ)
K = np.einsum('id,da->ia', X, WK)
V = np.einsum('id,da->ia', X, WV)

assert np.allclose(Q, X @ WQ)
print(f"Step 1: Q = X @ WQ  → shape {Q.shape}  ✓")
print(f"        K = X @ WK  → shape {K.shape}  ✓")
print(f"        V = X @ WV  → shape {V.shape}  ✓")

# ─────────────────────────────────────────────────────────────
# Step 2: Attention scores
#   S_ij = Q_ia K_ja / √d_k    →    einsum: 'ia,ja->ij'
#   d=a is dummy (contracted), i,j are free
# ─────────────────────────────────────────────────────────────
scale = np.sqrt(d_k)
S = np.einsum('ia,ja->ij', Q, K) / scale

assert np.allclose(S, Q @ K.T / scale)
print(f"\nStep 2: S = QK^T/√d_k  → shape {S.shape}  ✓")

# ─────────────────────────────────────────────────────────────
# Step 3: Softmax (applied per-row, over j for each i)
#   α_ij = exp(S_ij) / Σ_k exp(S_ik)
# ─────────────────────────────────────────────────────────────
alpha = softmax_rows(S)

assert np.allclose(alpha.sum(axis=1), 1.0)
print(f"Step 3: α = softmax(S)  → shape {alpha.shape}, rows sum to 1  ✓")

# ─────────────────────────────────────────────────────────────
# Step 4: Output
#   O_ia = α_ij V_ja    →    einsum: 'ij,ja->ia'
#   j is dummy (weighted sum over values)
# ─────────────────────────────────────────────────────────────
O = np.einsum('ij,ja->ia', alpha, V)

assert np.allclose(O, alpha @ V)
print(f"Step 4: O = α V  → shape {O.shape}  ✓")

print(f"\nComplete forward pass verified.")

In [ ]:
# Exercise 8: Solution ✅ — Backward Pass

dO = np.random.randn(T_seq, d_k)

print("═══ BACKWARD PASS ═══\n")

# ─────────────────────────────────────────────────────────────
# Step 4 backward: O_ia = α_ij V_ja
#
#   dL/dV_ja = α_ij dO_ia         →   einsum: 'ij,ia->ja'
#     (transpose of forward: α^T @ dO)
#
#   dL/dα_ij = dO_ia V_ja         →   einsum: 'ia,ja->ij'
#     (dO @ V^T)
# ─────────────────────────────────────────────────────────────
dV = np.einsum('ij,ia->ja', alpha, dO)
dalpha = np.einsum('ia,ja->ij', dO, V)

assert dV.shape == V.shape
assert dalpha.shape == alpha.shape
print(f"Step 4 backward:")
print(f"  dV = α^T @ dO        → shape {dV.shape}  ✓")
print(f"  dα = dO @ V^T        → shape {dalpha.shape}  ✓")

# ─────────────────────────────────────────────────────────────
# Step 3 backward: softmax gradient
#
#   dL/dS_ij = α_ij (dα_ij - Σ_k α_ik dα_ik)
#
# This uses the softmax Jacobian: J = diag(α_i) - α_i α_i^T
# applied per row.
# ─────────────────────────────────────────────────────────────
# For each row i: sum_k α_ik dα_ik (dot product of row i of α with row i of dα)
dot_per_row = np.einsum('ij,ij->i', alpha, dalpha)  # shape (T,)
dS = alpha * (dalpha - dot_per_row[:, None]) / scale

assert dS.shape == S.shape
print(f"\nStep 3 backward:")
print(f"  dS = α ⊙ (dα - Σ_k α_k dα_k) / √d_k  → shape {dS.shape}  ✓")

# ─────────────────────────────────────────────────────────────
# Step 2 backward: S_ij = Q_ia K_ja / √d_k
#
#   dL/dQ_ia = dS_ij K_ja         →   einsum: 'ij,ja->ia'
#     (dS @ K)
#
#   dL/dK_ja = dS_ij Q_ia         →   einsum: 'ij,ia->ja'
#     (dS^T @ Q)
# ─────────────────────────────────────────────────────────────
dQ = np.einsum('ij,ja->ia', dS, K)
dK = np.einsum('ij,ia->ja', dS, Q)

assert dQ.shape == Q.shape
assert dK.shape == K.shape
print(f"\nStep 2 backward:")
print(f"  dQ = dS @ K          → shape {dQ.shape}  ✓")
print(f"  dK = dS^T @ Q        → shape {dK.shape}  ✓")

# ─────────────────────────────────────────────────────────────
# Step 1 backward: Q_ia = X_id WQ_da
#
#   dL/dWQ_da = X_id dQ_ia        →   einsum: 'id,ia->da'
#     (X^T @ dQ)
#
#   Similarly for WK, WV.
# ─────────────────────────────────────────────────────────────
dWQ = np.einsum('id,ia->da', X, dQ)
dWK = np.einsum('id,ia->da', X, dK)
dWV = np.einsum('id,ia->da', X, dV)

assert dWQ.shape == WQ.shape
assert dWK.shape == WK.shape
assert dWV.shape == WV.shape
print(f"\nStep 1 backward:")
print(f"  dWQ = X^T @ dQ       → shape {dWQ.shape}  ✓")
print(f"  dWK = X^T @ dK       → shape {dWK.shape}  ✓")
print(f"  dWV = X^T @ dV       → shape {dWV.shape}  ✓")

print(f"\nComplete backward pass computed.")

In [ ]:
# Exercise 8: Solution ✅ — Gradient Verification

print("═══ GRADIENT VERIFICATION (Finite Differences) ═══\n")

eps = 1e-5

def attention_forward(X, WQ, WK, WV):
    """Full forward pass returning scalar loss = sum(O)."""
    Q = np.einsum('id,da->ia', X, WQ)
    K = np.einsum('id,da->ia', X, WK)
    V = np.einsum('id,da->ia', X, WV)
    S = np.einsum('ia,ja->ij', Q, K) / np.sqrt(WQ.shape[1])
    alpha = softmax_rows(S)
    O = np.einsum('ij,ja->ia', alpha, V)
    # Use L = Σ dO_ia O_ia so that dL/dO = dO (our upstream gradient)
    L = np.sum(dO * O)
    return L

# Verify dWQ
dWQ_num = np.zeros_like(WQ)
for i in range(WQ.shape[0]):
    for j in range(WQ.shape[1]):
        WQ_plus = WQ.copy();  WQ_plus[i, j] += eps
        WQ_minus = WQ.copy(); WQ_minus[i, j] -= eps
        dWQ_num[i, j] = (
            attention_forward(X, WQ_plus, WK, WV) -
            attention_forward(X, WQ_minus, WK, WV)
        ) / (2 * eps)

err_WQ = np.max(np.abs(dWQ - dWQ_num))
assert err_WQ < 1e-4, f"WQ gradient error too large: {err_WQ}"
print(f"dWQ: max error = {err_WQ:.2e}  ✓")

# Verify dWK
dWK_num = np.zeros_like(WK)
for i in range(WK.shape[0]):
    for j in range(WK.shape[1]):
        WK_plus = WK.copy();  WK_plus[i, j] += eps
        WK_minus = WK.copy(); WK_minus[i, j] -= eps
        dWK_num[i, j] = (
            attention_forward(X, WQ, WK_plus, WV) -
            attention_forward(X, WQ, WK_minus, WV)
        ) / (2 * eps)

err_WK = np.max(np.abs(dWK - dWK_num))
assert err_WK < 1e-4, f"WK gradient error too large: {err_WK}"
print(f"dWK: max error = {err_WK:.2e}  ✓")

# Verify dWV
dWV_num = np.zeros_like(WV)
for i in range(WV.shape[0]):
    for j in range(WV.shape[1]):
        WV_plus = WV.copy();  WV_plus[i, j] += eps
        WV_minus = WV.copy(); WV_minus[i, j] -= eps
        dWV_num[i, j] = (
            attention_forward(X, WQ, WK, WV_plus) -
            attention_forward(X, WQ, WK, WV_minus)
        ) / (2 * eps)

err_WV = np.max(np.abs(dWV - dWV_num))
assert err_WV < 1e-4, f"WV gradient error too large: {err_WV}"
print(f"dWV: max error = {err_WV:.2e}  ✓")

print(f"\n┌──────────────────────────────────────────────────┐")
print(f"│  ALL GRADIENTS VERIFIED against finite diffs  ✓  │")
print(f"└──────────────────────────────────────────────────┘")

print(f"\nIndex notation summary of attention backward pass:")
print(f"  dV_ja   = α_ij · dO_ia           einsum: 'ij,ia->ja'")
print(f"  dα_ij   = dO_ia · V_ja           einsum: 'ia,ja->ij'")
print(f"  dS_ij   = α_ij(dα_ij - Σ_k α_ik dα_ik) / √d_k")
print(f"  dQ_ia   = dS_ij · K_ja           einsum: 'ij,ja->ia'")
print(f"  dK_ja   = dS_ij · Q_ia           einsum: 'ij,ia->ja'")
print(f"  dWQ_da  = X_id · dQ_ia           einsum: 'id,ia->da'")
print(f"  dWK_da  = X_id · dK_ia           einsum: 'id,ia->da'")
print(f"  dWV_da  = X_id · dV_ia           einsum: 'id,ia->da'")

---

## Summary & Mastery Checklist

### Exercise Recap

| # | Topic | Key Skill | Level |
|---|-------|-----------|-------|
| 1 | Index Identification | Free vs dummy indices, einsum strings | ⭐ |
| 2 | Kronecker Delta | δ substitution, ε contraction rules | ⭐ |
| 3a | Quadratic Form Gradient | Product rule + δ_ij | ⭐⭐ |
| 3b | Frobenius Norm Gradient | Matrix calculus in indices | ⭐⭐ |
| 3c | Cross-Entropy + Softmax | Softmax Jacobian, chain rule | ⭐⭐ |
| 4 | Einsum Implementation | Translating notation to code | ⭐⭐ |
| 5 | Contraction Order | FLOP analysis, LoRA pattern | ⭐⭐ |
| 6 | Softmax Jacobian | Derivation + numerical verification | ⭐⭐ |
| 7 | Symmetry Analysis | Symmetric / antisymmetric proofs | ⭐⭐ |
| 8 | Full Attention Pass | Forward + backward + gradient check | ⭐⭐⭐ |

### Mastery Levels

**Level 1 — Fundamentals** (Exercises 1–2):
- [ ] Can identify free and dummy indices in any expression
- [ ] Can write the einsum string for any index expression
- [ ] Can simplify expressions using δ_ij and ε_ijk

**Level 2 — Applied** (Exercises 3–7):
- [ ] Can derive gradients mechanically using index notation
- [ ] Can translate any tensor operation to `np.einsum`
- [ ] Can analyse contraction order for computational efficiency
- [ ] Can derive and verify the softmax Jacobian
- [ ] Can prove symmetry properties from index expressions

**Level 3 — Production** (Exercise 8):
- [ ] Can implement complete attention forward pass in index notation
- [ ] Can derive the full backward pass using only index rules
- [ ] Can verify gradients with finite differences
- [ ] Can read and implement any paper equation using einsum